In [3]:
import pandas as pd
import numpy as np
import joblib
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv('../.env')

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

MODEL = "llama-3.3-70b-versatile"

print("Libraries imported")
print(f"Using model: {MODEL}")

Libraries imported
Using model: llama-3.3-70b-versatile


In [4]:
test_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Say hello in one sentence."}
    ]
)

print("Groq connection working:")
print(test_response.choices[0].message.content)

Groq connection working:
Hello, it's nice to meet you and I'm looking forward to chatting with you.


In [5]:
model     = joblib.load('../models/xgb_model.pkl')
X_test    = joblib.load('../models/X_test.pkl')
y_test    = joblib.load('../models/y_test.pkl')
explainer = joblib.load('../models/shap_explainer.pkl')
X_sample  = joblib.load('../models/X_sample.pkl')
y_sample  = joblib.load('../models/y_sample.pkl')

print("All files loaded successfully")
print(f"X_test shape:   {X_test.shape}")
print(f"X_sample shape: {X_sample.shape}")

All files loaded successfully
X_test shape:   (1272524, 11)
X_sample shape: (1000, 11)


In [6]:
import shap
import warnings
warnings.filterwarnings('ignore')

def get_shap_features(transaction_index):
    """
    Given a transaction index,
    returns top 5 SHAP features for that transaction
    """
    pos       = X_sample.index.get_loc(transaction_index)
    shap_vals = explainer.shap_values(X_sample.iloc[[pos]])[0]
    
    shap_df = pd.DataFrame({
        'feature':    X_sample.columns,
        'value':      X_sample.loc[transaction_index].values,
        'shap_value': shap_vals
    }).sort_values('shap_value', key=abs, ascending=False)
    
    return shap_df.head(5)

print("SHAP function ready")

SHAP function ready


In [7]:
def explain_fraud(transaction_index):
    """
    Full pipeline:
    transaction → prediction → SHAP → Groq → plain English
    """
    
    # Step 1: Get prediction
    transaction = X_sample.loc[[transaction_index]]
    fraud_prob  = model.predict_proba(transaction)[0][1]
    is_fraud    = fraud_prob > 0.3
    actual      = y_sample.loc[transaction_index]
    
    # Step 2: Get SHAP features
    top_features = get_shap_features(transaction_index)
    
    # Step 3: Build prompt
    features_text = ""
    for _, row in top_features.iterrows():
        direction = "increases" if row['shap_value'] > 0 else "decreases"
        features_text += f"- {row['feature']}: value={row['value']:.2f}, {direction} fraud risk (impact={abs(row['shap_value']):.3f})\n"
    
    prompt = f"""
You are a fraud analyst at a bank. Analyze this mobile money transaction.

Transaction Details:
- Fraud Probability: {fraud_prob:.4f} ({fraud_prob*100:.1f}%)
- Prediction: {"FRAUD" if is_fraud else "GENUINE"}
- Transaction Amount: {X_sample.loc[transaction_index, 'amount']:,.2f}
- Transaction Type: {X_sample.loc[transaction_index, 'type']}
- Sender Old Balance: {X_sample.loc[transaction_index, 'oldbalanceOrg']:,.2f}
- Sender New Balance: {X_sample.loc[transaction_index, 'newbalanceOrig']:,.2f}

Top factors driving this prediction:
{features_text}

Write a 3 sentence explanation in simple English for a bank employee.
Explain WHY this transaction is {"suspicious" if is_fraud else "legitimate"}.
Use the actual numbers from above.
Do not use technical ML terms like SHAP or XGBoost.
"""
    
    # Step 4: Call Groq
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a fraud analyst. Give clear, concise explanations based only on the evidence provided."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3,
        max_tokens=200
    )
    
    explanation = response.choices[0].message.content
    
    # Step 5: Return full result
    return {
        "transaction_index": transaction_index,
        "fraud_probability": round(fraud_prob, 4),
        "is_fraud":          bool(is_fraud),
        "actual_label":      int(actual),
        "top_features":      top_features.to_dict('records'),
        "explanation":       explanation
    }

print("explain_fraud function ready")

explain_fraud function ready


In [8]:
# Get a real fraud case from sample
fraud_indices = y_sample[y_sample == 1].index
fraud_idx     = fraud_indices[0]

print(f"Testing on fraud transaction: {fraud_idx}")
print()

result = explain_fraud(fraud_idx)

print(f"Fraud Probability: {result['fraud_probability']}")
print(f"Is Fraud:          {result['is_fraud']}")
print(f"Actual Label:      {result['actual_label']}")
print()
print("=" * 60)
print("PLAIN ENGLISH EXPLANATION:")
print("=" * 60)
print(result['explanation'])
print("=" * 60)

Testing on fraud transaction: 6020257

Fraud Probability: 1.0
Is Fraud:          True
Actual Label:      1

PLAIN ENGLISH EXPLANATION:
This mobile money transaction is suspicious because the sender's entire balance of 6,273,660.45 was transferred, leaving their new balance at 0.00, which is an unusual activity. The large transaction amount of 6,273,660.45, which is equal to the sender's old balance, raises concerns as it suggests that the sender is trying to quickly move all their funds. Additionally, the fact that the sender's old balance was exactly the same as the transaction amount, resulting in a 1.00 amount-to-balance ratio, increases the likelihood of this transaction being fraudulent.


In [9]:
# Get a genuine transaction
genuine_indices = y_sample[y_sample == 0].index
genuine_idx     = genuine_indices[0]

print(f"Testing on genuine transaction: {genuine_idx}")
print()

result_genuine = explain_fraud(genuine_idx)

print(f"Fraud Probability: {result_genuine['fraud_probability']}")
print(f"Is Fraud:          {result_genuine['is_fraud']}")
print()
print("=" * 60)
print("PLAIN ENGLISH EXPLANATION:")
print("=" * 60)
print(result_genuine['explanation'])
print("=" * 60)

Testing on genuine transaction: 1220251

Fraud Probability: 0.0
Is Fraud:          False

PLAIN ENGLISH EXPLANATION:
This transaction is considered legitimate because the sender's old and new balance are both 0.00, which suggests that this is not a suspicious transfer from a newly created account. The large transaction amount of 135,627.66 is also not suspicious because it is being sent to an account with a significant old balance of 196,307.15 and a new balance of 331,934.82, indicating that the recipient's account is active and able to handle large transactions. Overall, the combination of these factors, including the sender's and recipient's account balances, suggests that this transaction is genuine and has a low risk of being fraudulent.


In [10]:
print("Testing on 5 fraud transactions")
print("=" * 60)

for i, idx in enumerate(fraud_indices[:5]):
    result = explain_fraud(idx)
    print(f"\nFraud Case {i+1}:")
    print(f"Probability: {result['fraud_probability']}")
    print(f"Explanation: {result['explanation']}")
    print("-" * 60)

Testing on 5 fraud transactions

Fraud Case 1:
Probability: 1.0
Explanation: This mobile money transaction is suspicious because the sender's entire balance of 6,273,660.45 was transferred, leaving their new balance at 0.00, which is a unusual and high-risk activity. The fact that the transaction amount of 6,273,660.45 is equal to the sender's old balance, resulting in a 1.00 ratio, also increases the fraud risk. Additionally, the sender's old balance was very high at 6,273,660.45, which, combined with the other factors, raises concerns that this transaction may be fraudulent.
------------------------------------------------------------


In [11]:
import joblib

# Save sample results for testing API later
sample_results = []
for idx in fraud_indices[:3]:
    r = explain_fraud(idx)
    sample_results.append(r)

joblib.dump(sample_results, '../models/sample_explanations.pkl')

print("Sample explanations saved")
print()
print("=" * 50)
print("NOTEBOOK 05 COMPLETE")
print("=" * 50)
print("Groq LLM integration working")
print("Plain English explanations generated")
print("Ready for Notebook 06 - FastAPI")
print("=" * 50)

Sample explanations saved

NOTEBOOK 05 COMPLETE
Groq LLM integration working
Plain English explanations generated
Ready for Notebook 06 - FastAPI
